In [79]:
import numpy as np
import os
import data_ingestion_
from data_preparation_ import prepare_data_for_train_val_test_sets_, EmoDataset

the_project_info_ = data_ingestion_.get_toml_info_()
goemotions_ml_loc_ = the_project_info_["dataset_goemotions"]["goemotions_mldata_location_"]
train_ds_ = EmoDataset(goemotions_ml_loc_+"train.txt")
print(f"The text is -->> {train_ds_[200][0]},  and corresponding label is {train_ds_[200][1]}" )

The text is -->> Dirt bar downtown is pretty sweet.,  and corresponding label is 27


In [80]:
# -- pre trained model for emotions classification
from tokenizer_fn_ import pretrained_model_emoClassification, tokenize_function
the_text_ = train_ds_[200][0]
class_ = pretrained_model_emoClassification(the_text_)
print(class_)
tokens_ = tokenize_function(the_text_)
print(np.shape(tokens_['input_ids']))
print(np.shape(tokens_['attention_mask']))

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2588.02it/s]


[{'label': 'POSITIVE', 'score': 0.9998219609260559}]
(128,)
(128,)


In [81]:
from typing import Dict
import numpy as np
import ray

def add_dog_years(batch: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
    batch["age_in_dog_years"] = 7 * batch["age"]
    return batch

ds = (
    ray.data.from_items([
        {"name": "Luna", "age": 4},
        {"name": "Rory", "age": 14},
        {"name": "Scout", "age": 9},
    ])
    .map_batches(add_dog_years)
)
ds.take_batch()

# tokenized_datasets = train_ds_.map(tokenize_function, batched=True)
# tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

2026-03-31 05:33:43,965	INFO logging.py:392 -- Registered dataset logger for dataset dataset_8_0
2026-03-31 05:33:43,983	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_8_0. Full logs are in C:\Users\RAVISH~1\AppData\Local\Temp\ray\session_2026-03-27_00-09-10_935501_28836\logs\ray-data
2026-03-31 05:33:43,983	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_8_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(add_dog_years)] -> LimitOperator[limit=20]
2026-03-31 05:33:49,047	WARNING default_autoscaling_coordinator.py:134 -- Failed to send resource request for data-dataset_8_0. If this only happens transiently during network partition or CPU being overloaded, it's safe to ignore this error. If this error persists, file a GitHub issue.
Traceback (most recent call last):
  File "d:\Codes_\conda_\envs\distilbert_finetune\Lib\site-packages\ray\data\_internal\cluster_autoscaler\default_autoscaling_coordinator.py", line 98, in wrapper
    re

In [82]:
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = model.base_model

classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

enc = tokenizer(the_text_)
token_representations = base_model(torch.tensor(enc["input_ids"]).unsqueeze(0))[0][0]

print(enc["input_ids"])
print(tokenizer.decode(enc["input_ids"]))
print(f"Length: {len(enc['input_ids'])}")
print(token_representations.shape)



Loading weights: 100%|██████████| 104/104 [00:00<00:00, 4374.24it/s]


[101, 6900, 3347, 5116, 2003, 3492, 4086, 1012, 102]
[CLS] dirt bar downtown is pretty sweet. [SEP]
Length: 9
torch.Size([9, 768])


In [84]:
from model_definition_ import EmoModel
from transformers import AutoModelForMaskedLM
X = torch.tensor(enc["input_ids"]).unsqueeze(0).to('cpu')
attn = torch.tensor(enc["attention_mask"]).unsqueeze(0).to('cpu')
print(X.shape, attn.shape)
classifier_emomodel_ = EmoModel(AutoModelForMaskedLM.from_pretrained("distilroberta-base").base_model, 20)
hidd_ = classifier_emomodel_((X, attn))
print(hidd_)
print(hidd_.shape)

torch.Size([1, 9]) torch.Size([1, 9])


Loading weights: 100%|██████████| 106/106 [00:00<00:00, 3421.39it/s]
RobertaForMaskedLM LOAD REPORT from: distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tensor([[-0.0468,  0.0309,  0.0546,  0.1367, -0.0988,  0.0615, -0.0398,  0.1899,
         -0.1972,  0.0867, -0.0524, -0.0296,  0.0418, -0.0511,  0.0990, -0.0170,
          0.1191, -0.0831,  0.1059,  0.0313]], grad_fn=<AddmmBackward0>)
torch.Size([1, 20])


In [85]:
dataframe_emotions_, label_emotions_ = data_ingestion_.prepare_dataframe(the_project_info_)
print(label_emotions_)
desired_emotion_lab_ = label_emotions_

['admiration' 'amusement' 'anger' 'annoyance' 'approval' 'caring'
 'confusion' 'curiosity' 'desire' 'disappointment' 'disapproval' 'disgust'
 'embarrassment' 'excitement' 'fear' 'gratitude' 'grief' 'joy' 'love'
 'nervousness' 'optimism' 'pride' 'realization' 'relief' 'remorse'
 'sadness' 'surprise' 'neutral']


In [86]:
from tokenizers import ByteLevelBPETokenizer
from tokenizers.processors import BertProcessing
class TokenizersCollateFn:
    def __init__(self, max_tokens=512):
      ## RoBERTa uses BPE tokenizer similar to GPT
      t = ByteLevelBPETokenizer(
          "tokenizer/vocab.json",
          "tokenizer/merges.txt"
      )
      t._tokenizer.post_processor = BertProcessing(
          ("</s>", t.token_to_id("</s>")),
          ("<s>", t.token_to_id("<s>")),
      )
      t.enable_truncation(max_tokens)
      t.enable_padding(length=max_tokens, pad_id=t.token_to_id("<pad>"))
      self.tokenizer = t

    def __call__(self, batch):
      encoded = self.tokenizer.encode_batch([x[0] for x in batch])
      sequences_padded = torch.tensor([enc.ids for enc in encoded])
      attention_masks_padded = torch.tensor([enc.attention_mask for enc in encoded])
      labels = torch.tensor([x[1] for x in batch])

      return (sequences_padded, attention_masks_padded), labels

In [88]:
## Methods required by PyTorchLightning
import pytorch_lightning as pl
from torch import nn
from typing import List
from functools import lru_cache
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW as AdamW
from transformers import get_linear_schedule_with_warmup


class TrainingModule(pl.LightningModule):
    def __init__(self, hparams):
        super().__init__()
        self.model = EmoModel(AutoModelForSequenceClassification.from_pretrained("distilroberta-base").base_model, len(desired_emotion_lab_))
        self.loss = nn.CrossEntropyLoss() ## combines LogSoftmax() and NLLLoss()
        #self.hparams = hparams
        self.hparams.update(vars(hparams))

    def step(self, batch, step_name="train"):
        X, y = batch
        loss = self.loss(self.forward(X), y)
        loss_key = f"{step_name}_loss"
        tensorboard_logs = {loss_key: loss}

        return { ("loss" if step_name == "train" else loss_key): loss, 'log': tensorboard_logs,
               "progress_bar": {loss_key: loss}}

    def forward(self, X, *args):
        return self.model(X, *args)

    def training_step(self, batch, batch_idx):
        return self.step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self.step(batch, "val")

    def validation_end(self, outputs: List[dict]):
        loss = torch.stack([x["val_loss"] for x in outputs]).mean()
        return {"val_loss": loss}

    def test_step(self, batch, batch_idx):
        return self.step(batch, "test")

    def train_dataloader(self):
        return self.create_data_loader(self.hparams.train_path, shuffle=True)

    def val_dataloader(self):
        return self.create_data_loader(self.hparams.val_path)

    def test_dataloader(self):
        return self.create_data_loader(self.hparams.test_path)

    def create_data_loader(self, ds_path: str, shuffle=False):
        return DataLoader(
                    EmoDataset(ds_path),
                    batch_size=self.hparams.batch_size,
                    shuffle=shuffle,
                    collate_fn=TokenizersCollateFn()
        )

    @lru_cache()
    def total_steps(self):
        return len(self.train_dataloader()) // self.hparams.accumulate_grad_batches * self.hparams.epochs

    def configure_optimizers(self):
        ## use AdamW optimizer -- faster approach to training NNs
        ## read: https://www.fast.ai/2018/07/02/adam-weight-decay/
        optimizer = AdamW(self.model.parameters(), lr=self.hparams.lr)
        lr_scheduler = get_linear_schedule_with_warmup(
                    optimizer,
                    num_warmup_steps=self.hparams.warmup_steps,
                    num_training_steps=self.total_steps(),
        )
        return [optimizer], [{"scheduler": lr_scheduler, "interval": "step"}]

In [90]:
from torch.utils.data import DataLoader
from torch_lr_finder import LRFinder

desired_batch_size, real_batch_size = 32, 4
accumulation_steps = desired_batch_size // real_batch_size

dataset = train_path
real_batch_size = 50
# Beware of the `batch_size` used by `DataLoader`
trainloader = DataLoader(dataset, batch_size=real_batch_size, shuffle=True)
classifier_emomodel_ = EmoModel(AutoModelForMaskedLM.from_pretrained("distilroberta-base").base_model, len(desired_emotion_lab_))

model = classifier_emomodel_
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(module.parameters(), lr=5e-7) ## lower bound LR

# (Optional) With this setting, `amp.scale_loss()` will be adopted automatically.
# model, optimizer = amp.initialize(model, optimizer, opt_level='O1')

lr_finder = LRFinder(model, optimizer, criterion, device="cuda")
lr_finder.range_test(trainloader, end_lr=10, num_iter=100, step_mode="exp", accumulation_steps=accumulation_steps)
lr_finder.plot()
lr_finder.reset()

Loading weights: 100%|██████████| 106/106 [00:00<00:00, 4439.79it/s]
RobertaForMaskedLM LOAD REPORT from: distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx